# Notebook 02 — Exploratory Data Analysis

## Purpose

Describe the complete raw Facebook corpus and, separately, the manually annotated subset. This notebook does not train topic models or classifiers.

## Inputs

Checked outputs from `outputs/01_data_preparation/20261018_content_review/`:

- `raw_corpus_working.csv` — 5,292 comments
- `annotation_linked.csv` — 1,199 annotated comments
- `post_url_crosswalk.csv` — 19 source posts
- `content_evidence_review.csv` — qualified descriptions for POST_17 and POST_18

## Rules

- Counts of comments and counts of source posts are labelled separately.
- `collection_date` is when the comment file was collected. It is not the source post's publication date.
- `publication_date_register` is the researcher-verified date stored in the publication workbook.
- Manual labels describe only the 1,199 annotated comments. They are not labels for the other comments.
- POST_17 and POST_18 descriptions remain qualified. Their original Facebook captions were not recovered.


## 0. Load the checked data-preparation outputs

Read the working corpus and the linked annotation table. Do not fit a model in this notebook.


In [1]:
from pathlib import Path
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent, Path(r"D:\NIRA_CASE_STUDY")]
ROOT = None
for candidate in candidates:
    if (candidate / "NIRA_Facebook_Datasets_5292.csv").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Project root not found.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config.project_config import EXPECTED_ANNOTATION_N, EXPECTED_N_POSTS, EXPECTED_RAW_N

RUN_IN = ROOT / "outputs" / "01_data_preparation" / "20261018_content_review"
RUN_ID = "20261018_eda"
OUT = ROOT / "outputs" / "02_eda" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures"
FIG.mkdir(exist_ok=True)

required = [
    "raw_corpus_working.csv",
    "annotation_linked.csv",
    "post_url_crosswalk.csv",
    "content_evidence_review.csv",
]
for name in required:
    path = RUN_IN / name
    if not path.exists():
        raise FileNotFoundError(path)

raw = pd.read_csv(RUN_IN / "raw_corpus_working.csv")
ann = pd.read_csv(RUN_IN / "annotation_linked.csv")
posts = pd.read_csv(RUN_IN / "post_url_crosswalk.csv")
content_review = pd.read_csv(RUN_IN / "content_evidence_review.csv")

if len(raw) != EXPECTED_RAW_N or len(ann) != EXPECTED_ANNOTATION_N or len(posts) != EXPECTED_N_POSTS:
    raise ValueError(f"Unexpected sizes: raw={len(raw)}, ann={len(ann)}, posts={len(posts)}")
if raw["record_key"].duplicated().any() or ann["record_key"].duplicated().any():
    raise ValueError("record_key is not unique.")
if not set(ann["record_key"]).issubset(set(raw["record_key"])):
    raise ValueError("Some annotated keys are missing from the raw working corpus.")

raw["collection_date_parsed"] = pd.to_datetime(raw["collection_date_parsed"])
raw["publication_date_register"] = pd.to_datetime(raw["publication_date_register"])
posts["publication_date"] = pd.to_datetime(posts["publication_date"])
print("Raw comments:", len(raw))
print("Annotated comments:", len(ann))
print("Source posts:", len(posts))
print("Output directory:", OUT)


Raw comments: 5292
Annotated comments: 1199
Source posts: 19
Output directory: D:\NIRA_CASE_STUDY\outputs\02_eda\20261018_eda


### Interpretation

The checks confirm 5,292 raw comments, 1,199 annotated comments, and 19 posts. The annotated keys are a subset of the raw keys. Descriptions below keep those two denominators separate.


## 1. Raw corpus structure

These figures describe all 5,292 comments. They do not describe the annotated labels.


In [2]:
raw["collection_day"] = raw["collection_date_parsed"].dt.date
post_view = (
    raw.groupby(["post_id", "account_name_register"], as_index=False)
    .agg(
        n_comments=("record_key", "size"),
        n_annotated=("is_annotated", "sum"),
        collection_date=("collection_date_parsed", "min"),
        n_collection_timestamps=("collection_date_parsed", "nunique"),
        publication_date_register=("publication_date_register", "min"),
        median_chars=("comment_char_len", "median"),
        short_under_30=("comment_char_len", lambda s: int((s < 30).sum())),
        phone_like=("phone_like_flag", "sum"),
        arabic_script=("arabic_script_flag", "sum"),
        styled_latin=("styled_latin_flag", "sum"),
    )
    .sort_values("publication_date_register")
)
post_view["annotated_share_of_post"] = post_view["n_annotated"] / post_view["n_comments"]
post_view["share_of_raw_corpus"] = post_view["n_comments"] / len(raw)

if int(post_view["n_comments"].sum()) != len(raw) or int(post_view["n_annotated"].sum()) != len(ann):
    raise ValueError("Post totals do not match the corpus.")
if not (post_view["n_collection_timestamps"] == 1).all():
    raise ValueError("A post has more than one collection timestamp.")

print(post_view.assign(
    collection_date=post_view["collection_date"].dt.strftime("%Y-%m-%d %H:%M"),
    publication_date_register=post_view["publication_date_register"].dt.strftime("%Y-%m-%d"),
)[[
    "post_id", "n_comments", "n_annotated", "collection_date", "publication_date_register", "median_chars"
]].to_string(index=False))
print("Comment share of the largest post:", round(post_view["share_of_raw_corpus"].max(), 3))
print("Annotated share within posts:", round(post_view["annotated_share_of_post"].min(), 3), "to", round(post_view["annotated_share_of_post"].max(), 3))


post_id  n_comments  n_annotated  collection_date publication_date_register  median_chars
POST_02         660          150 2026-07-24 20:27                2023-09-16          46.0
POST_16          92           21 2026-07-24 20:45                2023-09-16          50.5
POST_05         320           73 2026-07-24 20:54                2023-10-02          44.0
POST_19          52           12 2026-07-24 21:25                2024-01-30          58.5
POST_14          97           22 2026-07-24 19:48                2025-09-18          68.0
POST_12         160           36 2026-07-24 19:24                2025-09-26          46.0
POST_03         642          146 2026-08-08 23:21                2025-10-02          72.0
POST_10         200           45 2026-08-08 22:53                2025-10-18          62.5
POST_01        1013          229 2026-08-08 22:45                2025-10-27          57.0
POST_09         212           48 2026-08-08 23:05                2025-11-03          81.5
POST_04   

### Interpretation

Comment volume is highly uneven across posts. One post contributes 1,013 comments, while the smallest contributes 52. A description of the corpus is therefore influenced by the largest posts.


## 2. Collection dates and publication dates

The left-hand pattern below is the collection timeline. The right-hand pattern is the register publication timeline. They answer different questions.


In [3]:
collection_days = raw.groupby("collection_day").size().rename("n_comments").reset_index()
publication_years = (
    raw.groupby(raw["publication_date_register"].dt.year)
    .size()
    .rename("n_comments")
    .reset_index()
    .rename(columns={"publication_date_register": "publication_year"})
)
print("Collection days (comment counts):")
print(collection_days.to_string(index=False))
print("Register publication years (comment counts, using publication_date_register):")
print(publication_years.to_string(index=False))
print(
    "Publication dates run from",
    posts["publication_date"].min().date().isoformat(),
    "to",
    posts["publication_date"].max().date().isoformat(),
)


Collection days (comment counts):
collection_day  n_comments
    2026-07-23         505
    2026-07-24        2720
    2026-08-08        2067
Register publication years (comment counts, using publication_date_register):
 publication_year  n_comments
             2023        1072
             2024          52
             2025        3169
             2026         999
Publication dates run from 2023-09-16 to 2026-07-14


### Interpretation

Collection dates and publication dates are different clocks. The comments were collected on three days in July and August 2026: 505, 2,720, and 2,067 comments. The source posts were published from 2023 through 2026. A comment collected in 2026 can sit under a post published in 2023.


## 3. Comment length and text characteristics

Length is measured on `comment_original` for the 5,292 raw comments. The annotated column is the same comments restricted to the 1,199 matched records.


In [4]:
annotated_mask = raw["is_annotated"].astype(bool)
length_summary = pd.DataFrame({
    "group": ["raw_corpus", "annotated_subset"],
    "n_comments": [len(raw), int(annotated_mask.sum())],
    "min_chars": [int(raw["comment_char_len"].min()), int(raw.loc[annotated_mask, "comment_char_len"].min())],
    "median_chars": [float(raw["comment_char_len"].median()), float(raw.loc[annotated_mask, "comment_char_len"].median())],
    "mean_chars": [float(raw["comment_char_len"].mean()), float(raw.loc[annotated_mask, "comment_char_len"].mean())],
    "max_chars": [int(raw["comment_char_len"].max()), int(raw.loc[annotated_mask, "comment_char_len"].max())],
    "under_30_chars": [int((raw["comment_char_len"] < 30).sum()), int((raw.loc[annotated_mask, "comment_char_len"] < 30).sum())],
    "over_280_chars": [int((raw["comment_char_len"] > 280).sum()), int((raw.loc[annotated_mask, "comment_char_len"] > 280).sum())],
    "with_newline": [int(raw["has_newline"].astype(bool).sum()), int(raw.loc[annotated_mask, "has_newline"].astype(bool).sum())],
    "arabic_script": [int(raw["arabic_script_flag"].astype(bool).sum()), int(raw.loc[annotated_mask, "arabic_script_flag"].astype(bool).sum())],
    "styled_latin": [int(raw["styled_latin_flag"].astype(bool).sum()), int(raw.loc[annotated_mask, "styled_latin_flag"].astype(bool).sum())],
    "phone_like": [int(raw["phone_like_flag"].astype(bool).sum()), int(raw.loc[annotated_mask, "phone_like_flag"].astype(bool).sum())],
})
print(length_summary.to_string(index=False))


           group  n_comments  min_chars  median_chars  mean_chars  max_chars  under_30_chars  over_280_chars  with_newline  arabic_script  styled_latin  phone_like
      raw_corpus        5292         15          55.0   79.311980       1430            1024             142           335              7             5          11
annotated_subset        1199         15          57.0   80.086739        919             240              34            70              4             2           2


### Interpretation

The raw comments have a median length of 55 characters, and the annotated comments have a median of 57. About 1,024 raw comments are shorter than 30 characters. Short text limits what a topic model or a classifier can see in a single comment.


## 4. Manual labels

Every percentage in this section uses 1,199 annotated comments as the denominator. These are not percentages of the 5,292-comment corpus.


In [5]:
label_frames = []
for field in ["policy_relevance", "feedback_type", "topic", "sentiment", "actor"]:
    counts = ann[field].value_counts().rename_axis("label").reset_index(name="n_comments")
    counts.insert(0, "field", field)
    counts["share_of_annotated"] = counts["n_comments"] / len(ann)
    label_frames.append(counts)
label_counts = pd.concat(label_frames, ignore_index=True)
print(label_counts.to_string(index=False))

topic_sentiment = pd.crosstab(ann["topic"], ann["sentiment"], margins=True)
feedback_relevance = pd.crosstab(ann["feedback_type"], ann["policy_relevance"], margins=True)
print("Topic by sentiment (annotated comments):")
print(topic_sentiment.to_string())
print("Feedback type by policy relevance (annotated comments):")
print(feedback_relevance.to_string())

text_match = ann["text_match_status"].value_counts().rename_axis("text_match_status").reset_index(name="n_comments")
text_match["share_of_annotated"] = text_match["n_comments"] / len(ann)
print(text_match.to_string(index=False))


           field                         label  n_comments  share_of_annotated
policy_relevance                      relevant         821            0.684737
policy_relevance                  not_relevant         237            0.197665
policy_relevance                     uncertain         141            0.117598
   feedback_type                     complaint         317            0.264387
   feedback_type                         other         232            0.193495
   feedback_type                       support         172            0.143453
   feedback_type                      question         162            0.135113
   feedback_type                       request         153            0.127606
   feedback_type                     criticism          98            0.081735
   feedback_type                    experience          44            0.036697
   feedback_type                   information          21            0.017515
           topic                         other      

### Interpretation

Among the 1,199 labeled comments, `relevant` is 821, `complaint` is 317, topic `other` is 390, and `negative` is 510. The crosstabs show how these labels occur together. They are not estimates for the unlabeled comments.


## 5. Qualified post descriptions

POST_17 and POST_18 have working descriptions in the content-evidence review. Those descriptions are not recovered Facebook captions.


In [6]:
review_view = content_review[[
    "post_id", "publication_date", "resolution_status", "confidence", "working_content_description", "remaining_limitation"
]].copy()
print(review_view.to_string(index=False))


post_id publication_date                                  resolution_status                                                                      confidence                                                                                                                          working_content_description                                                                                                                            remaining_limitation
POST_17       2025-12-26      LIKELY IDENTIFIED — topic-level corroboration   High for the likely topic; not confirmed as the identical Facebook media item                                                   Likely report/repost about the Somali Embassy in Uganda beginning to issue NIRA national ID cards. Facebook URL and publication date are researcher-verified, but the original Facebook caption/video was not directly retrievable in this review.
POST_18       2026-03-19 PROBABLE — inferred from comments and date context Moderate for an Eid-relate

### Interpretation

The content-review table keeps POST_17 and POST_18 qualified. At the time of this run their Facebook captions had not been read back from the platform. A later researcher screenshot recorded those captions, and that later record is the one to cite for the wording.


## 6. Figures

Each figure states whether the count is comments from the raw corpus or from the annotated subset.


In [7]:
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = False

ordered = post_view.sort_values("n_comments", ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(ordered["post_id"], ordered["n_comments"], label="Raw comments (n = 5,292)")
ax.barh(ordered["post_id"], ordered["n_annotated"], label="Annotated comments (n = 1,199)")
ax.set_xlabel("Number of comments")
ax.set_ylabel("Source post")
ax.set_title("Comments by source post")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "comments_by_post.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(raw["comment_char_len"], bins=40, color="#4C78A8")
ax.set_xlabel("Comment length (characters)")
ax.set_ylabel("Number of comments")
ax.set_title("Comment length in the raw corpus (n = 5,292)")
fig.tight_layout()
fig.savefig(FIG / "comment_length_raw.png", dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(collection_days["collection_day"].astype(str), collection_days["n_comments"], color="#4C78A8")
axes[0].set_title("Comments by collection day")
axes[0].set_xlabel("Collection date")
axes[0].set_ylabel("Number of comments")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(publication_years["publication_year"].astype(str), publication_years["n_comments"], color="#F58518")
axes[1].set_title("Comments by register publication year")
axes[1].set_xlabel("Publication year in the register")
axes[1].set_ylabel("Number of comments")
fig.tight_layout()
fig.savefig(FIG / "collection_day_and_publication_year.png", dpi=150)
plt.close(fig)

label_fields = ["policy_relevance", "feedback_type", "topic", "sentiment"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, field in zip(axes.ravel(), label_fields):
    part = label_counts[label_counts["field"] == field].sort_values("n_comments")
    ax.barh(part["label"], part["n_comments"], color="#54A24B")
    ax.set_xlabel("Annotated comments")
    ax.set_title(field.replace("_", " "))
fig.suptitle("Manual labels (denominator = 1,199 annotated comments)")
fig.tight_layout()
fig.savefig(FIG / "annotated_label_distributions.png", dpi=150)
plt.close(fig)
print("Saved figures:", sorted(p.name for p in FIG.glob('*.png')))


Saved figures: ['annotated_label_distributions.png', 'collection_day_and_publication_year.png', 'comment_length_raw.png', 'comments_by_post.png']


### Interpretation

The figures separate posts, comment length, collection day, publication year, and the annotated label counts. No topic model or classifier is fitted here.


## 7. Save and verify

The saved tables are the descriptive inputs for later notebooks. This run does not create topics or predicted labels.


In [8]:
def save_df(frame: pd.DataFrame, name: str) -> None:
    path = OUT / name
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Missing output: {path}")
    print("OK", name, len(frame))

post_out = post_view.copy()
post_out["collection_date"] = post_out["collection_date"].dt.strftime("%Y-%m-%d %H:%M:%S")
post_out["publication_date_register"] = post_out["publication_date_register"].dt.strftime("%Y-%m-%d")
save_df(post_out, "eda_post_summary.csv")
save_df(length_summary, "eda_length_summary.csv")
save_df(label_counts, "eda_label_counts.csv")
save_df(topic_sentiment.reset_index(), "eda_topic_by_sentiment.csv")
save_df(feedback_relevance.reset_index(), "eda_feedback_by_relevance.csv")
save_df(collection_days.assign(collection_day=collection_days["collection_day"].astype(str)), "eda_collection_days.csv")
save_df(publication_years, "eda_publication_years.csv")
save_df(text_match, "eda_text_match.csv")
save_df(review_view, "eda_content_review_excerpt.csv")

for figure in FIG.glob("*.png"):
    if figure.stat().st_size < 1000:
        raise RuntimeError(f"Figure looks empty: {figure}")

findings = {
    "run_id": RUN_ID,
    "input_run": "20261018_content_review",
    "raw_comments": int(len(raw)),
    "annotated_comments": int(len(ann)),
    "source_posts": int(len(posts)),
    "largest_post_id": str(post_view.sort_values("n_comments").iloc[-1]["post_id"]),
    "largest_post_comments": int(post_view["n_comments"].max()),
    "annotated_share_of_raw": len(ann) / len(raw),
    "annotated_share_by_post_min": float(post_view["annotated_share_of_post"].min()),
    "annotated_share_by_post_max": float(post_view["annotated_share_of_post"].max()),
    "collection_days": collection_days.assign(collection_day=collection_days["collection_day"].astype(str)).to_dict(orient="records"),
    "publication_year_comment_counts": publication_years.to_dict(orient="records"),
    "raw_median_chars": float(raw["comment_char_len"].median()),
    "raw_under_30": int((raw["comment_char_len"] < 30).sum()),
    "dominant_labels": {
        field: {
            "label": str(label_counts.loc[label_counts["field"] == field, "label"].iloc[0]),
            "n_comments": int(label_counts.loc[label_counts["field"] == field, "n_comments"].iloc[0]),
        }
        for field in ["policy_relevance", "feedback_type", "topic", "sentiment"]
    },
}
(OUT / "eda_findings.json").write_text(json.dumps(findings, indent=2), encoding="utf-8")
print("Findings written.")
print("Raw median characters:", findings["raw_median_chars"])
print("Dominant annotated labels:", findings["dominant_labels"])


OK eda_post_summary.csv 19
OK eda_length_summary.csv 2
OK eda_label_counts.csv 28
OK eda_topic_by_sentiment.csv 9
OK eda_feedback_by_relevance.csv 9
OK eda_collection_days.csv 3
OK eda_publication_years.csv 4
OK eda_text_match.csv 3
OK eda_content_review_excerpt.csv 2
Findings written.
Raw median characters: 55.0
Dominant annotated labels: {'policy_relevance': {'label': 'relevant', 'n_comments': 821}, 'feedback_type': {'label': 'complaint', 'n_comments': 317}, 'topic': {'label': 'other', 'n_comments': 390}, 'sentiment': {'label': 'negative', 'n_comments': 510}}


### Interpretation

The tables and `eda_findings.json` store the counts used by the figures. A later notebook should read these files rather than retype the counts.


## 8. What this description supports

- The raw corpus is uneven across posts. One post contributes the largest share of comments. Later corpus-wide results need to keep that concentration visible.
- Annotated comments are spread across every post at roughly 22–23 percent. The selection rule is still not documented.
- Many comments are short. That will limit both topic terms and supervised classification.
- The largest annotated categories are a starting description of the labelled subset only.
- Collection dates cluster on three days in July and August 2026. Register publication dates run from 2023 to 2026. Those timelines must not be substituted for each other.
- No topic model or classifier has been fitted in this notebook.
